In [ ]:
import pandas as pd

df = pd.read_csv('/content/id_metadata.csv', sep = '\t')
df1 = pd.read_csv('/content/id_genres.csv', sep = '\t')

In [ ]:
# left: сохраняем весь каталог, треки без жанров получают NaN (train_bm25 превращает их в пустые)
result = df.merge(df1, on='id', how='left')

In [ ]:
result = result[['spotify_id','genres']]
items = result.drop_duplicates(subset='spotify_id').reset_index(drop=True)


In [ ]:
items

In [ ]:
"""BM25 query/document retrieval over spotify_id and genres.

Dependencies: numpy, pandas, scipy. Genres are whole tokens: 'alternative rock'
is one genre, not two words. genres cells may be lists of strings, comma-separated
strings, or strings representing Python/JSON lists. Null genres become empty.
Uses positive smoothed IDF. Each distinct query genre contributes once.
"""
import ast
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix


def train_bm25(items: pd.DataFrame, k1: float = 1.2, b: float = 0.75,
               feature_col: str = 'genres') -> dict:
    """Build a BM25 index from spotify_id and the selected feature column."""
    if not np.isfinite(k1) or k1 <= 0 or not np.isfinite(b) or not 0 <= b <= 1:
        raise ValueError('Require finite k1 > 0 and 0 <= b <= 1')
    if not {'spotify_id', feature_col}.issubset(items.columns):
        raise ValueError(f'Expected spotify_id and {feature_col} columns')
    if items.empty or items.spotify_id.isna().any() or items.spotify_id.duplicated().any():
        raise ValueError('Expected nonempty data with unique, non-null spotify_id')
    vocab, rows, cols, frequencies = {}, [], [], []
    lengths = np.zeros(len(items), dtype=float)
    for row, genres in enumerate(items[feature_col]):
        if isinstance(genres, str):
            genres = genres.strip()
            genres = (ast.literal_eval(genres) if genres.startswith('[')
                      else genres.split(','))
        elif genres is None or genres is pd.NA or (
            np.isscalar(genres) and pd.isna(genres)
        ):
            genres = []
        if not isinstance(genres, (list, tuple, set, np.ndarray)):
            raise ValueError(f'Invalid genres at row {row}')
        if any(not isinstance(g, str) for g in genres):
            raise ValueError(f'Genre names must be strings at row {row}')
        terms = [
    word
    for genre in genres
    for word in genre.strip().casefold().split()]
        lengths[row] = len(terms)
        for term, tf in Counter(terms).items():
            column = vocab.setdefault(term, len(vocab))
            rows.append(row)
            cols.append(column)
            frequencies.append(tf)
    if not vocab:
        raise ValueError('No genres available to index')
    counts = coo_matrix((np.asarray(frequencies, dtype=float), (rows, cols)),
                        shape=(len(items), len(vocab))).tocsc()
    df = np.diff(counts.indptr)
    idf = np.log1p((len(items) - df + 0.5) / (df + 0.5))
    weighted = counts.tocoo()
    length_norm = 1 - b + b * lengths / lengths.mean()
    weighted.data = (weighted.data * (k1 + 1) /
                     (weighted.data + k1 * length_norm[weighted.row]) *
                     idf[weighted.col])
    return {'index': weighted.tocsc(), 'vocab': vocab,
            'item_ids': items['spotify_id'].to_numpy(copy=True)}


def infer_bm25(model: dict, query, top_k: int = 20, exclude_id=None) -> dict:
    """Return {spotify_id: score} in descending order for a genre query.

    query: list of genre strings, or one genre string. Multiword genres stay
    intact. Unknown genres are ignored. No overlap returns {}. exclude_id can
    remove the source track. This is query/document BM25, not vector similarity.
    """
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 0:
        raise ValueError('top_k must be a nonnegative integer')
    if isinstance(query, str):
        query = [query]
    if not isinstance(query, (list, tuple, set, np.ndarray)):
        raise ValueError('query must be a genre string or a sequence of genres')
    if any(not isinstance(g, str) for g in query):
        raise ValueError('Query genres must be strings')
    terms = {
      word
      for part in query
      for word in part.strip().casefold().split()
  }
    columns = [model['vocab'][g] for g in terms if g in model['vocab']]
    if top_k == 0 or not columns:
        return {}
    # One BM25 contribution per matching query term; only candidate document
    # weights are used. Never multiply two BM25-weighted track vectors.
    scores = np.asarray(model['index'][:, columns].sum(axis=1)).ravel()
    valid = scores > 0
    if exclude_id is not None:
        valid &= model['item_ids'] != exclude_id
    candidates = np.flatnonzero(valid)
    order = np.lexsort((candidates, -scores[candidates]))[:top_k]
    return {model['item_ids'][i]: float(scores[i]) for i in candidates[order]}


In [ ]:
model = train_bm25(items, feature_col='genres')

In [ ]:
ids = infer_bm25(model,'some pop music', top_k=20)

In [ ]:
items[items['spotify_id'].isin(ids.keys())]

In [ ]:
model